### 1. Import the libraries and read the Excel worksheet

Working from a copy makes all edits traceable and keeps the supplied workbook intact. An inpatient number is an identifier, so read it as text. This file contains one worksheet named `responsivenes`. The code tries two nearby locations for the file and prints its dimensions, column names, and data types; it does not display patient records.

In [1]:
from pathlib import Path
import pandas as pd

source_candidates = [Path('responsivenes.xlsx'), Path('upload/responsivenes.xlsx')]
source_path = next((p for p in source_candidates if p.exists()), None)
if source_path is None:
    raise FileNotFoundError('Place responsivenes.xlsx beside this notebook or in upload/.')

df_original = pd.read_excel(
    source_path, sheet_name='responsivenes', dtype={'inpatient_number': 'string'}
)
df_clean = df_original.copy(deep=True)
print('Starting shape:', df_original.shape)
print('Source columns:', df_original.columns.tolist())
print('Starting data types:')
print(df_original.dtypes.to_string())

Starting shape: (2008, 6)
Source columns: ['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs']
Starting data types:
inpatient_number    string[python]
eye_opening                  int64
verbal_response              int64
movement                     int64
consciousness               object
gcs                          int64


### 2. Review the columns before deleting anything

First check whether a field is unrelated or exactly derived from another. All six source columns have a purpose here: the ID supports joins, the three component scores describe different responses, `consciousness` is a recorded category, and `gcs` is a useful total. We verify below that `gcs` equals the sum of its parts. Keep that verified total in the cleaned file because it is commonly useful for summaries. For a later prediction model, avoid using the total *alongside* its components as if they were four independent measurements. The code records that no source column is dropped at this cleaning stage.

In [2]:
source_columns = df_clean.columns.tolist()
expected_columns = [
    'inpatient_number', 'eye_opening', 'verbal_response',
    'movement', 'consciousness', 'gcs'
]
if source_columns != expected_columns:
    raise ValueError(f'Unexpected columns; inspect this source: {source_columns}')

columns_removed = []
print('Source columns removed:', columns_removed)
print('Reason: each source field is retained for identification, description, or validation.')

Source columns removed: []
Reason: each source field is retained for identification, description, or validation.


### 3. Rename inconsistent column headers

Standardize headers to a short, predictable naming style: lowercase words separated by underscores. This avoids code failures caused by extra spaces or inconsistent capitalization. These six column names are already standardized, so this check does not rename any header here.

In [3]:
old_headers = df_clean.columns.tolist()
df_clean.columns = (df_clean.columns.astype(str).str.strip().str.lower()
                    .str.replace(r'\s+', '_', regex=True))
header_changes = sum(a != b for a, b in zip(old_headers, df_clean.columns))
print('Column names changed:', header_changes)
print('Final header names:', df_clean.columns.tolist())

Column names changed: 0
Final header names: ['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs']


### 4. Identify duplicate records and duplicate IDs

**Why:** A repeated full row would count the same data twice in charts. Remove *exact* duplicate rows only. Check repeated `inpatient_number` values separately, since a repeated ID could reflect a real second encounter and should not be discarded by guesswork. This file has neither kind of duplicate.

In [4]:
exact_duplicates = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
duplicate_ids = int(df_clean['inpatient_number'].duplicated().sum())

print('Exact duplicate rows removed:', exact_duplicates)
print('Repeated inpatient_number values:', duplicate_ids)
if duplicate_ids:
    raise ValueError('Repeated patient IDs need source review before export.')

Exact duplicate rows removed: 0
Repeated inpatient_number values: 0


### 5. Count missing values before deciding how to handle them

Inspect nulls by column first. Strip surrounding spaces from the two text fields, and treat empty strings as missing. Neither `No` nor `NR` here can be assumed to be a missing clinical score or consciousness label. There are no missing values in this workbook, so no imputation or row removal is necessary. If a future copy contains a blank, the code pauses for review.

In [5]:
df_clean['inpatient_number'] = df_clean['inpatient_number'].str.strip()
df_clean['consciousness'] = df_clean['consciousness'].astype('string').str.strip()
for column in ['inpatient_number', 'consciousness']:
    df_clean[column] = df_clean[column].replace('', pd.NA)

missing_by_column = df_clean.isna().sum()
missing_percent = (df_clean.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing_by_column,
    'missing_percent': missing_percent
})
print(missing_report.to_string())
if missing_by_column.any():
    raise ValueError('Missing values require column-specific source review; none are guessed.')

                  missing_count  missing_percent
inpatient_number              0              0.0
eye_opening                   0              0.0
verbal_response               0              0.0
movement                      0              0.0
consciousness                 0              0.0
gcs                           0              0.0


### 6. Standardize the recorded `consciousness` labels

Remove extra internal spaces and normalize capitalization to the four labels already in this file: `Clear`, `ResponsiveToSound`, `ResponsiveToPain`, and `Nonresponsive`. A category outside this list stops the notebook for inspection rather than being assigned an invented label. The supplied file has four recognized categories and no formatting changes.

In [6]:
label_map = {
    'clear': 'Clear',
    'responsivetosound': 'ResponsiveToSound',
    'responsivetopain': 'ResponsiveToPain',
    'nonresponsive': 'Nonresponsive',
}
labels_before = df_clean['consciousness'].copy()
label_keys = (df_clean['consciousness']
              .str.replace(r'\s+', '', regex=True)
              .str.casefold())
standard_labels = label_keys.map(label_map)
if standard_labels.isna().any():
    raise ValueError('An unrecognized consciousness label needs manual review.')
df_clean['consciousness'] = standard_labels.astype('string')
labels_changed = int((labels_before != df_clean['consciousness']).sum())

print('Labels changed by formatting:', labels_changed)
print('Category frequencies:')
print(df_clean['consciousness'].value_counts().to_string())

Labels changed by formatting: 0
Category frequencies:
consciousness
Clear                1974
ResponsiveToSound      19
Nonresponsive          11
ResponsiveToPain        4


### 7. Remove numeric noise and check allowed score values

Convert the score fields to numeric and reject invalid entries. GCS component ranges are eye opening **1–4**, verbal response **1–5**, and movement **1–6**; the total is **3–15**. Entries that are not numbers, are fractional, or lie outside these ranges need source review. No score is guessed or replaced. All rows in this file pass.

In [7]:
valid_ranges = {
    'eye_opening': (1, 4),
    'verbal_response': (1, 5),
    'movement': (1, 6),
    'gcs': (3, 15),
}
for column, (lower, upper) in valid_ranges.items():
    # Nonnumeric text becomes NaN for counting, not for silent export.
    numeric = pd.to_numeric(df_clean[column], errors='coerce')
    invalid = numeric.isna() | numeric.mod(1).ne(0) | ~numeric.between(lower, upper)
    print(f'{column}: invalid={int(invalid.sum())}; '
          f'observed range={numeric.min()} to {numeric.max()}')
    if invalid.any():
        raise ValueError(f'Inspect {column}: {int(invalid.sum())} invalid score(s).')
    df_clean[column] = numeric.astype('int64')

eye_opening: invalid=0; observed range=1 to 4
verbal_response: invalid=0; observed range=1 to 5
movement: invalid=0; observed range=1 to 6
gcs: invalid=0; observed range=3 to 15


### 8. Verify the derived GCS total

Remove fields that can be recreated by calculation. Here, `gcs` should equal `eye_opening + verbal_response + movement`; calculate it and compare all rows before deciding whether it is trustworthy. Zero rows disagree. We keep the verified total in the cleaned export for easy reporting. A redundancy principle is applied to the optional feature list below, where only the three components are used; the patient ID, label, review flag, and derived total are excluded as predictors.

In [8]:
gcs_from_components = (
    df_clean['eye_opening'] + df_clean['verbal_response'] + df_clean['movement']
)
gcs_mismatches = int(gcs_from_components.ne(df_clean['gcs']).sum())
print('Rows where recorded GCS differs from E + V + M:', gcs_mismatches)
if gcs_mismatches:
    raise ValueError('Resolve GCS mismatches with the source before export.')

# An optional analysis view, not an additional copy of the cleaned dataset:
component_features = df_clean[['eye_opening', 'verbal_response', 'movement']].copy()
print('Nonredundant score features for a model:', component_features.columns.tolist())

Rows where recorded GCS differs from E + V + M: 0
Nonredundant score features for a model: ['eye_opening', 'verbal_response', 'movement']


### 9. Check whether identical scores have different labels

cleans inconsistent categories; some patients with identical **eye, verbal, and movement** scores have different `consciousness` labels. A score combination may lack clinical context that affected the recorded label, so assigning a new label would be a guess. Mark all records in these combinations with `consciousness_review_flag=True` while leaving the original label intact. There are **two** such score combinations affecting **29** records. This flag indicates a question for the data owner, not a confirmed medical error.

In [9]:
components = ['eye_opening', 'verbal_response', 'movement']
labels_per_pattern = df_clean.groupby(components)['consciousness'].transform('nunique')
df_clean['consciousness_review_flag'] = labels_per_pattern.gt(1)

review_groups = (df_clean.loc[df_clean['consciousness_review_flag']]
                 .groupby(components)
                 .agg(rows=('gcs', 'size'), different_labels=('consciousness', 'nunique'))
                 .reset_index())
print('Score patterns needing review:', len(review_groups))
print('Rows flagged for review:', int(df_clean['consciousness_review_flag'].sum()))
print(review_groups.to_string(index=False))

Score patterns needing review: 2
Rows flagged for review: 29
 eye_opening  verbal_response  movement  rows  different_labels
           1                1         1    13                 4
           3                3         5    16                 2


### 10. Show the missingness summary, as in the reference notebook

Plot the percentage missing by column; the table below gives those percentages here. Because **all six input columns have 0% missing values**, there are no positive-height bars to plot; the conditional chart code runs only if a later version contains blanks. This avoids showing an empty graph as if it were an insight.

In [10]:
print('Missing percentage by input column:')
print(missing_percent.to_string())

if missing_percent.gt(0).any():
    import matplotlib.pyplot as plt
    missing_percent[missing_percent.gt(0)].sort_values().plot.barh(
        title='Percentage of missing values by column', figsize=(8, 4)
    )
    plt.xlabel('Percent missing')
    plt.tight_layout()
    plt.show()
else:
    print('No missing-value chart is needed: every input column is complete.')

Missing percentage by input column:
inpatient_number    0.0
eye_opening         0.0
verbal_response     0.0
movement            0.0
consciousness       0.0
gcs                 0.0
No missing-value chart is needed: every input column is complete.


### 11. Confirm the cleaning results before export

Compare the input and cleaned dimensions, verify unique IDs and no missing cells, and summarize every cleaning decision. In this particular file, validation mattered more than correction: no source rows or source values were changed; one review column was added. The original six fields remain available for analysis.

In [11]:
assert len(df_clean) == len(df_original) - exact_duplicates
assert df_clean['inpatient_number'].is_unique
assert not df_clean[expected_columns].isna().any().any()
assert len(df_clean.columns) == len(expected_columns) + 1

audit = pd.DataFrame([
    ('Input rows', len(df_original), 'Starting worksheet'),
    ('Rows removed as exact duplicates', exact_duplicates, 'Zero in this file'),
    ('Rows retained', len(df_clean), 'No patient records lost'),
    ('Columns removed', len(columns_removed), 'All original fields retained'),
    ('Column headers changed', header_changes, 'Names already standardized'),
    ('Missing cells in original fields', int(missing_by_column.sum()), 'No imputation'),
    ('Duplicate patient IDs', duplicate_ids, 'None found'),
    ('Category formatting changes', labels_changed, 'No labels recoded'),
    ('GCS arithmetic mismatches', gcs_mismatches, 'All totals verified'),
    ('Rows flagged for label review', int(df_clean['consciousness_review_flag'].sum()),
     'Recorded labels preserved'),
], columns=['Cleaning check', 'Count', 'Decision'])
print(audit.to_string(index=False))
print('Cleaned shape:', df_clean.shape)

                  Cleaning check  Count                     Decision
                      Input rows   2008           Starting worksheet
Rows removed as exact duplicates      0            Zero in this file
                   Rows retained   2008      No patient records lost
                 Columns removed      0 All original fields retained
          Column headers changed      0   Names already standardized
Missing cells in original fields      0                No imputation
           Duplicate patient IDs      0                   None found
     Category formatting changes      0            No labels recoded
       GCS arithmetic mismatches      0          All totals verified
   Rows flagged for label review     29    Recorded labels preserved
Cleaned shape: (2008, 7)


### 12. Export and verify the cleaned dataset

Save a CSV for the hackathon workflow. It has the six source fields plus `consciousness_review_flag`. Re-open the exported file and confirm row count, IDs, columns, and review flags. The notebook itself is the reproducible record of the cleaning steps.

In [12]:
output_path = Path('responsivenes_cleaned.csv')
df_clean.to_csv(output_path, index=False)

check = pd.read_csv(output_path, dtype={'inpatient_number': 'string'})
assert len(check) == len(df_clean)
assert check.columns.tolist() == df_clean.columns.tolist()
assert check['inpatient_number'].equals(df_clean['inpatient_number'])
assert int(check['consciousness_review_flag'].sum()) == int(
    df_clean['consciousness_review_flag'].sum()
)
print(f'Exported and verified: {output_path.name}')
print(f'Rows: {len(check):,}; columns: {len(check.columns)}')

Exported and verified: responsivenes_cleaned.csv
Rows: 2,008; columns: 7
